# Get annotation assignments

To account for possible changes to volunteer availability, I reduced the sample size to 482 citing opinions, spanning ~22K records of citing-cited pairs, from which we will be able to produce the final authoritative treatments by all subsequent citing opinion for 15 cited scotus opinions. This should be doable in 2 weeks and we can get another set of eyes on all these for a 2nd round annotation after the 1st round is completed.

# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import json
import random
import os

# Sample from the final df to produce a small sample set for expert annotation

In [3]:
df = pd.read_json("../experiments_603/data/scotus_citing_cited.json")
df.head()

,citing_cluster_id,cited_cluster_id,citation_depth,citing_cited,citing_url,citing_opinions,citing_filenames,cited_url,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations,cited_target
0,4877180,108291,5,4877180-108291,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/108291/o...,Wyandotte Chemicals,Ohio v. Wyandotte Chemicals Corp.,OHIO v. WYANDOTTE CHEMICALS CORP. Et Al.,"[2 ERC (BNA) 1331, 1 Envtl. L. Rep. (Envtl. La...",0
1,4877180,95159,4,4877180-95159,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/95159/lo...,,Louisiana v. Texas,Louisiana v. Texas,"[1900 U.S. LEXIS 1715, 44 L. Ed. 347, 20 S. Ct...",0
2,4877180,109452,4,4877180-109452,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/109452/a...,AZ v. NM,Arizona v. New Mexico,Arizona v. New Mexico,"[1976 U.S. LEXIS 117, 425 U.S. 794, 96 S. Ct. ...",0
3,4877180,108523,3,4877180-108523,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/108523/i...,Milwaukee,Illinois v. City of Milwaukee,"ILLINOIS v. CITY OF MILWAUKEE, WISCONSIN, Et Al.","[4 ERC (BNA) 1001, 2 Envtl. L. Rep. (Envtl. La...",0
4,4877180,110972,3,4877180-110972,https://www.courtlistener.com/opinion/4877180/...,"[{'opinion_id': 4680959, 'opinion_api': 'https...",[4877180_010combined.txt],https://www.courtlistener.com/opinion/110972/t...,TX v. NM,Texas v. New Mexico,Texas v. New Mexico,"[51 U.S.L.W. 4805, 1983 U.S. LEXIS 67, 462 U.S...",0


In [4]:
keep = list(df[df["cited_target"] == 1]["cited_cluster_id"].unique())
len(keep)

879

In [5]:
random.seed(42)
keep_sampled = random.sample(keep, 15)
len(keep_sampled)

15

In [6]:
cited_sampled = df[df["cited_cluster_id"].isin(keep_sampled)]
citing_sampled = cited_sampled["citing_cluster_id"].unique()

In [7]:
depth_sampled = df[df["citing_cluster_id"].isin(citing_sampled)]
len(depth_sampled)

21925

In [8]:
depth_sampled["citing_cited"].nunique()

21925

In [9]:
depth_sampled["citing_cluster_id"].nunique()

482

In [10]:
depth_sampled[depth_sampled["cited_cluster_id"].isin(keep_sampled)][["citing_cluster_id", "cited_cluster_id"]].value_counts("cited_cluster_id")

cited_cluster_id
103012     226
103172      53
109016      47
96357       46
102091      27
103976      25
105513      20
1447641     16
101492      15
105837      15
103548       6
86090        5
105004       5
84798        1
536051       1
Name: count, dtype: int64

In [11]:
eda_cols = ['citing_cluster_id', 'cited_cluster_id', 'citation_depth', 'citing_cited', 'cited_target']

for col in eda_cols:
    print("----------")
    print(depth_sampled[col].nunique())
    display(depth_sampled[col].value_counts())

----------
482


citing_cluster_id
106249     294
106366     227
106548     196
106267     186
106170     184
          ... 
105527       4
100915       2
105845       2
2620779      2
105846       1
Name: count, Length: 482, dtype: int64

----------
11627


cited_cluster_id
103012     226
85272       67
103172      53
109016      47
96357       46
          ... 
321053       1
603502       1
103555       1
253427       1
2620773      1
Name: count, Length: 11627, dtype: int64

----------
110


citation_depth
2      9607
1      4644
4      3044
6       955
3       895
       ... 
251       1
59        1
90        1
76        1
161       1
Name: count, Length: 110, dtype: int64

----------
21925


citing_cited
106775-99123       1
110765-386007      1
110765-108612      1
110765-103832      1
110765-109836      1
                  ..
104139-3569190     1
104139-95459       1
104139-103572      1
104139-3631017     1
2620779-2620773    1
Name: count, Length: 21925, dtype: int64

----------
2


cited_target
0    18868
1     3057
Name: count, dtype: int64

## Check how many in the sampled dataset are overruled

The ratio is in-line with expectation.

In [12]:
data = pd.read_csv("../experiments_501/data/output_dataset.csv")

data[data["cited_cluster_id"].isin(keep_sampled)].value_counts("overruled")

overruled
no     12
yes     4
Name: count, dtype: int64

In [13]:
data[data["cited_cluster_id"].isin(keep_sampled)].value_counts("citing_cluster_id")

citing_cluster_id
103915    2
93904     1
103869    1
104380    1
105525    1
107748    1
108350    1
109252    1
110212    1
111308    1
111404    1
112040    1
112258    1
112640    1
118317    1
Name: count, dtype: int64

## Save for expert annotation

In [14]:
depth_sampled.to_json("data/scotus_citing_cited_sampled.json")

In [15]:
annotation = depth_sampled[['citing_cluster_id', 'citing_url', 'cited_cluster_id', 'cited_url', 'cited_case_name_short', 'cited_case_name', 'cited_citations']]
annotation.head()

,citing_cluster_id,citing_url,cited_cluster_id,cited_url,cited_case_name_short,cited_case_name,cited_citations
214,106775,https://www.courtlistener.com/opinion/106775/b...,99123,https://www.courtlistener.com/opinion/99123/oe...,Oetjen,Oetjen v. Central Leather Co.,"[1918 U.S. LEXIS 1548, 62 L. Ed. 726, 38 S. Ct..."
215,106775,https://www.courtlistener.com/opinion/106775/b...,1498457,https://www.courtlistener.com/opinion/1498457/...,Bernstein,Bernstein v. Van Heyghen Freres Societe Anonyme,"[1947 U.S. App. LEXIS 3211, 163 F.2d 246]"
216,106775,https://www.courtlistener.com/opinion/106775/b...,94759,https://www.courtlistener.com/opinion/94759/un...,Underhill,Underhill v. Hernandez,"[1897 U.S. LEXIS 1721, 42 L. Ed. 456, 18 S. Ct..."
217,106775,https://www.courtlistener.com/opinion/106775/b...,99124,https://www.courtlistener.com/opinion/99124/ri...,Ricaud,Ricaud v. American Metal Co.,"[1918 U.S. LEXIS 1549, 62 L. Ed. 733, 38 S. Ct..."
218,106775,https://www.courtlistener.com/opinion/106775/b...,94252,https://www.courtlistener.com/opinion/94252/hi...,Hilton,Hilton v. Guyot,"[1895 U.S. LEXIS 2294, 40 L. Ed. 95, 16 S. Ct...."


# Assign to experts based on their availability

In [ ]:
more_than_ten = []
five_to_ten = []
less_than_five = []

In [17]:
capacity_map = {}

more_than_ten_max = {name: 10*60*2 for name in more_than_ten} # ~10hrs/week for 2 weeks
five_to_ten_max = {name: 6.5*60*2 for name in five_to_ten} # ~6.5hrs/week for 2 weeks
less_than_five_max = {name: 3*60*2 for name in less_than_five} # ~3hrs/week for 2 weeks

capacity_map.update(more_than_ten_max)
capacity_map.update(five_to_ten_max)
capacity_map.update(less_than_five_max)

In [18]:
assignment = annotation.groupby(["citing_cluster_id"]).size().reset_index(name="num_authorities") #assume each authority takes 1 minute to annotate
assignment.head()

,citing_cluster_id,num_authorities
0,1722,49
1,2146,9
2,2147,5
3,93904,8
4,94610,30


In [19]:
for i, row in assignment.iterrows():
    for name, available in capacity_map.items():
        if available >= row['num_authorities']:
            assignment.at[i, 'expert'] = name
            capacity_map[name] -= row['num_authorities']
            break

In [21]:
assert len(assignment[assignment["expert"].isna()]) == 0

In [22]:
assert set(assignment["expert"].unique()) == set(more_than_ten + five_to_ten + less_than_five)

In [23]:
double_check = assignment.groupby('expert')['num_authorities'].sum().reset_index()

for expert, max_mins in more_than_ten_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

for expert, max_mins in five_to_ten_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

for expert, max_mins in less_than_five_max.items():
    assert double_check[double_check["expert"] == expert]["num_authorities"].iloc[0] <= max_mins

In [24]:
assignment_dict = dict(zip(assignment['citing_cluster_id'], assignment['expert']))

In [25]:
for citing_cluster_id in annotation["citing_cluster_id"].unique():
    annotate = annotation[annotation["citing_cluster_id"] == citing_cluster_id].copy()
    expert = assignment_dict[citing_cluster_id]
    annotate["expert"] = expert
    annotate["expert_label"] = None
    num_cited = annotate["cited_cluster_id"].nunique()

    expert_folder = f"data/annotations/{expert}"
    os.makedirs(expert_folder, exist_ok=True)
    
    annotate.to_csv(f"{expert_folder}/{num_cited}_{citing_cluster_id}.csv", index=False)